In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ \mathbb{R} \to \mathbb{R}, \quad p(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} $$

$$ \mathbb{R^n} \to \mathbb{R^n}, \quad p(\mathbf{z}) = (p(z_1), p(z_2), \dots, p(z_n)) $$

$$ \text{Derivative} $$

$$ \frac{dp}{dz_i} = 1 - p_i^2 $$

$$ \frac{dp}{dz} = \mathbf{1} - \mathbf{p} \odot \mathbf{p} $$

$$ \text{Jacobian} $$

$$
J_p(\mathbf{z}) =
\begin{bmatrix}
    \frac{dp}{dz_1} & 0 & \cdots & 0 \\
    0 & \frac{dp}{dz_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dp}{dz_n}
\end{bmatrix}
$$

$$ d\mathbf{p} = J_p(\mathbf{z}) \, d\mathbf{z} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, d\mathbf{z} \right \rangle =
\left \langle \frac{dF}{dp}, d\mathbf{p} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dp}, d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{dp}, J_p(\mathbf{z}) \, d\mathbf{z} \right \rangle =
\left \langle J_p(\mathbf{z}) ^\top \, \frac{dF}{dp}, d\mathbf{z} \right \rangle \implies
$$

$$ 
\frac{dF}{dz} = 
J_p(\mathbf{z})^\top \, \frac{dF}{dp} = 
\frac{dp}{dz} \odot \frac{dF}{dp} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore


def tanh(z: tr.Tensor) -> tr.Tensor:
    """Hyperbolic tangent function."""
    
    return 2 * sigmoid.sigmoid(2 * z) - 1
    

class TanhFunction(autograd.Function):
    """Custom autograd function with `forward` and `backward` passes for the `tanh`."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        p = tanh(z)  
        ctx.save_for_backward(p)
        return p

    @staticmethod
    def backward(ctx, dF_dp: tr.Tensor) -> tuple[tr.Tensor, ]:
        (p, ) = ctx.saved_tensors

        dp_dz = 1 - p * p
        dF_dz = dp_dz * dF_dp

        return (dF_dz, )
    

class Tanh(nn.Module):
    """Custom module for the `tanh`."""

    def forward(self, z: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(z)


In [ ]:
# Unit tests

def test_basic() -> None:
    assert tanh(1000) == approx(tr.tensor(1.0))
    assert tanh(-1000) == approx(tr.tensor(-1.0))
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1) == approx(tr.tensor(0.76159))
    assert tanh(-1) == approx(tr.tensor(-0.76159))


def test_gradcheck(samples) -> None:
    def func(z: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(z)

    z = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (z, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    z = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    z1 = z.clone().detach().requires_grad_(True)
    p1 = Tanh()(z1)
    p1.sum().backward()

    z2 = z.clone().detach().requires_grad_(True)
    p2 = tr.tanh(z2)
    p2.sum().backward()

    assert p1 == approx(p2)
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)
    
    test_compare(1)
    test_compare(10)
    test_compare(100)